## time-series exploration

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("inflowOutflowForecast").getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "America/Toronto")
print("Spark version:", spark.version)

# Point to the parent 'rides' folder – Spark will discover the ride_year partitions
rides_df = spark.read.parquet("../../data/silver/rides/", unionByName=True)

# Check schema and partition columns
rides_df.printSchema()
rides_df.show(5)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/28 10:09:14 WARN Utils: Your hostname, users-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 10.249.165.215 instead (on interface en0)
26/03/28 10:09:14 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/28 10:09:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.1


root
 |-- end_station_key: string (nullable = true)
 |-- start_station_key: string (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_district: string (nullable = true)
 |-- start_station_latitude: double (nullable = true)
 |-- start_station_longitude: double (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_district: string (nullable = true)
 |-- end_station_latitude: double (nullable = true)
 |-- end_station_longitude: double (nullable = true)
 |-- start_time_ms: timestamp (nullable = true)
 |-- end_time_ms: timestamp (nullable = true)
 |-- start_station_name_norm: string (nullable = true)
 |-- end_station_name_norm: string (nullable = true)
 |-- start_coord_key: string (nullable = true)
 |-- end_coord_key: string (nullable = true)
 |-- start_canonical_station_id: string (nullable = true)
 |-- end_canonical_station_id: string (nullable = true)
 |-- ride_year: integer (nullable = true)



+--------------------+--------------------+--------------------+----------------------+----------------------+-----------------------+--------------------+--------------------+--------------------+---------------------+-------------------+-------------------+-----------------------+---------------------+--------------------+--------------------+--------------------------+------------------------+---------+
|     end_station_key|   start_station_key|  start_station_name|start_station_district|start_station_latitude|start_station_longitude|    end_station_name|end_station_district|end_station_latitude|end_station_longitude|      start_time_ms|        end_time_ms|start_station_name_norm|end_station_name_norm|     start_coord_key|       end_coord_key|start_canonical_station_id|end_canonical_station_id|ride_year|
+--------------------+--------------------+--------------------+----------------------+----------------------+-----------------------+--------------------+--------------------+----

In [2]:
### getting other live data 
import requests
import json
import sqlite3
import csv
from datetime import datetime
import os

# ==================== CONFIGURATION ====================

DB_FILE = "bikeshare.db"
CSV_EXPORT_DIR = "csv_exports"

# Feed URLs
FEEDS = [
    {"name": "station_information", "url": "https://gbfs.velobixi.com/gbfs/2-2/en/station_information.json"},
    {"name": "system_alerts",        "url": "https://gbfs.velobixi.com/gbfs/2-2/en/system_alerts.json"},
    {"name": "system_information",   "url": "https://gbfs.velobixi.com/gbfs/2-2/en/system_information.json"},
    {"name": "vehicle_types",        "url": "https://gbfs.velobixi.com/gbfs/2-2/en/vehicle_types.json"},
    {"name": "station_status",       "url": "https://gbfs.velobixi.com/gbfs/2-2/en/station_status.json"}
]

# Table definitions: each entry contains:
#   - table_name: SQL table name
#   - columns: list of column names (in the order they should be inserted)
#   - extractor: function that returns a list of rows to insert, each row a list of values
#               (will receive the feed's data dict and the global last_updated timestamp)
#   - primary_key: (optional) used for INSERT OR REPLACE or for conflict handling

TABLE_DEFS = {
    "station_information": {
        "table_name": "stations",
        "columns": [
            "station_id", "external_id", "name", "short_name", "lat", "lon",
            "rental_methods", "capacity", "electric_bike_surcharge_waiver",
            "is_charging", "eightd_has_key_dispenser", "has_kiosk", "last_updated"
        ],
        "primary_key": "station_id",
        "extractor": lambda data, last_updated: [
            (
                s.get("station_id"),
                s.get("external_id"),
                s.get("name"),
                s.get("short_name"),
                s.get("lat"),
                s.get("lon"),
                json.dumps(s.get("rental_methods", [])),      # store as JSON string
                s.get("capacity"),
                1 if s.get("electric_bike_surcharge_waiver", False) else 0,
                1 if s.get("is_charging", False) else 0,
                1 if s.get("eightd_has_key_dispenser", False) else 0,
                1 if s.get("has_kiosk", False) else 0,
                last_updated
            )
            for s in data["data"]["stations"]
        ]
    },
    "system_alerts": {
        "table_name": "system_alerts",
        "columns": [
            "alert_id", "type", "times", "url", "summary", "description",
            "alert_last_updated", "feed_last_updated"
        ],
        "primary_key": "alert_id",
        "extractor": lambda data, last_updated: [
            (
                a.get("alert_id"),
                a.get("type"),
                json.dumps(a.get("times", [])),               # store times as JSON
                a.get("url"),
                a.get("summary"),
                a.get("description"),
                a.get("last_updated"),
                last_updated
            )
            for a in data["data"].get("alerts", [])
        ]
    },
    "system_information": {
        "table_name": "system_information",
        "columns": [
            "system_id", "language", "name", "short_name", "operator", "url",
            "purchase_url", "start_date", "phone_number", "email", "timezone",
            "license_url", "last_updated"
        ],
        "primary_key": "system_id",
        "extractor": lambda data, last_updated: [
            (
                data["data"].get("system_id"),
                data["data"].get("language"),
                data["data"].get("name"),
                data["data"].get("short_name"),
                data["data"].get("operator"),
                data["data"].get("url"),
                data["data"].get("purchase_url"),
                data["data"].get("start_date"),
                data["data"].get("phone_number"),
                data["data"].get("email"),
                data["data"].get("timezone"),
                data["data"].get("license_url"),
                last_updated
            )
        ]
    },
    "vehicle_types": {
        "table_name": "vehicle_types",
        "columns": [
            "vehicle_type_id", "name", "form_factor", "propulsion_type",
            "max_range_meters", "name_length", "last_updated"
        ],
        "primary_key": "vehicle_type_id",
        "extractor": lambda data, last_updated: [
            (
                vt.get("vehicle_type_id"),
                vt.get("name"),
                vt.get("form_factor"),
                vt.get("propulsion_type"),
                vt.get("max_range_meters"),
                vt.get("name_length"),
                last_updated
            )
            for vt in data["data"].get("vehicle_types", [])
        ]
    },
    "station_status": {
        "table_name": "station_status",
        "columns": [
            "station_id", "num_bikes_available", "num_ebikes_available",
            "num_bikes_disabled", "num_docks_available", "num_docks_disabled",
            "is_installed", "is_renting", "is_returning", "last_reported",
            "vehicle_types_available", "vehicle_docks_available", "feed_last_updated"
        ],
        "primary_key": ["station_id", "feed_last_updated"],   # composite key
        "extractor": lambda data, last_updated: [
            (
                s.get("station_id"),
                s.get("num_bikes_available", 0),
                s.get("num_ebikes_available", 0),
                s.get("num_bikes_disabled", 0),
                s.get("num_docks_available", 0),
                s.get("num_docks_disabled", 0),
                1 if s.get("is_installed", False) else 0,
                1 if s.get("is_renting", False) else 0,
                1 if s.get("is_returning", False) else 0,
                s.get("last_reported", last_updated),
                json.dumps(s.get("vehicle_types_available", [])),
                json.dumps(s.get("vehicle_docks_available", [])),
                last_updated
            )
            for s in data["data"]["stations"]
        ]
    }
}

# ==================== HELPER FUNCTIONS ====================

def ensure_table_schema(conn, table_name, columns, primary_key):
    """Create table if it doesn't exist, or add missing columns."""
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name=?", (table_name,))
    table_exists = cursor.fetchone() is not None

    if not table_exists:
        # Create table
        col_defs = [f"{col} TEXT" for col in columns]   # all columns as TEXT for simplicity
        if primary_key:
            if isinstance(primary_key, list):
                pk_def = f"PRIMARY KEY ({', '.join(primary_key)})"
            else:
                pk_def = f"PRIMARY KEY ({primary_key})"
            col_defs.append(pk_def)
        create_sql = f"CREATE TABLE {table_name} ({', '.join(col_defs)})"
        cursor.execute(create_sql)
        print(f"  🆕 Created table {table_name}")
    else:
        # Check for missing columns and add them
        cursor.execute(f"PRAGMA table_info({table_name})")
        existing_columns = {row[1] for row in cursor.fetchall()}
        for col in columns:
            if col not in existing_columns:
                cursor.execute(f"ALTER TABLE {table_name} ADD COLUMN {col} TEXT")
                print(f"  ➕ Added column {col} to {table_name}")
    conn.commit()

def insert_rows(conn, table_name, columns, rows, primary_key):
    """Insert or replace rows using a generic statement."""
    cursor = conn.cursor()
    if not rows:
        return
    placeholders = ", ".join(["?"] * len(columns))
    if primary_key:
        # Use INSERT OR REPLACE (works for single or composite PK)
        sql = f"INSERT OR REPLACE INTO {table_name} ({', '.join(columns)}) VALUES ({placeholders})"
    else:
        sql = f"INSERT INTO {table_name} ({', '.join(columns)}) VALUES ({placeholders})"
    cursor.executemany(sql, rows)
    conn.commit()

def export_table_to_csv(conn, table_name):
    """Export a table to CSV (unchanged)."""
    cursor = conn.cursor()
    cursor.execute(f"SELECT * FROM {table_name}")
    rows = cursor.fetchall()
    if not rows:
        print(f"  ⚠️  Table '{table_name}' is empty, skipping CSV export.")
        return
    # Get column names
    cursor.execute(f"PRAGMA table_info({table_name})")
    columns = [col[1] for col in cursor.fetchall()]
    os.makedirs(CSV_EXPORT_DIR, exist_ok=True)
    csv_file = os.path.join(CSV_EXPORT_DIR, f"{table_name}.csv")
    with open(csv_file, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(columns)
        writer.writerows(rows)
    print(f"  📄 Exported {len(rows)} rows to {csv_file}")

def export_all_tables_to_csv(conn):
    """Export all tables defined in TABLE_DEFS."""
    print(f"\n📁 Exporting tables to CSV in '{CSV_EXPORT_DIR}' directory...")
    for table_key in TABLE_DEFS:
        export_table_to_csv(conn, TABLE_DEFS[table_key]["table_name"])

# ==================== MAIN PROCESSING ====================

def main():
    print(f"Starting data fetch at {datetime.now()}")

    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()

    # Ensure tables exist with latest schema
    for table_key, defs in TABLE_DEFS.items():
        ensure_table_schema(conn, defs["table_name"], defs["columns"], defs.get("primary_key"))

    # Process each feed
    for feed in FEEDS:
        feed_name = feed["name"]
        print(f"\n📡 Fetching {feed_name}...")
        try:
            response = requests.get(feed["url"], timeout=10)
            if response.status_code == 200:
                data = response.json()
                last_updated = data.get("last_updated", int(datetime.now().timestamp()))

                # Find matching table definition
                table_key = None
                for key in TABLE_DEFS:
                    if key == feed_name:
                        table_key = key
                        break
                if table_key is None:
                    print(f"  ⚠️  No table definition for feed {feed_name}, skipping.")
                    continue

                defs = TABLE_DEFS[table_key]
                rows = defs["extractor"](data, last_updated)
                if rows:
                    insert_rows(conn, defs["table_name"], defs["columns"], rows, defs.get("primary_key"))
                    print(f"  ✅ Inserted/updated {len(rows)} rows into {defs['table_name']}")
                else:
                    print(f"  ℹ️  No data extracted from {feed_name}")
            else:
                print(f"  ❌ Failed: HTTP {response.status_code}")
        except Exception as e:
            print(f"  ❌ Error: {str(e)}")

    # Export all tables to CSV
    export_all_tables_to_csv(conn)

    conn.close()
    print(f"\n✅ All feeds processed. Database: {DB_FILE}")
    print(f"📁 CSV files saved in '{CSV_EXPORT_DIR}' folder")
    print(f"Completed at {datetime.now()}")

if __name__ == "__main__":
    main()

Starting data fetch at 2026-03-28 10:09:54.237807

📡 Fetching station_information...
  ✅ Inserted/updated 456 rows into stations

📡 Fetching system_alerts...
  ✅ Inserted/updated 39 rows into system_alerts

📡 Fetching system_information...
  ✅ Inserted/updated 1 rows into system_information

📡 Fetching vehicle_types...
  ✅ Inserted/updated 5 rows into vehicle_types

📡 Fetching station_status...
  ✅ Inserted/updated 456 rows into station_status

📁 Exporting tables to CSV in 'csv_exports' directory...
  📄 Exported 456 rows to csv_exports/stations.csv
  📄 Exported 39 rows to csv_exports/system_alerts.csv
  📄 Exported 1 rows to csv_exports/system_information.csv
  📄 Exported 5 rows to csv_exports/vehicle_types.csv
  📄 Exported 2989 rows to csv_exports/station_status.csv

✅ All feeds processed. Database: bikeshare.db
📁 CSV files saved in 'csv_exports' folder
Completed at 2026-03-28 10:10:02.459087


In [3]:
### Point to the parent 'station direct matching' folder – 

station_mapping_df = spark.read.parquet("../../data/silver/station_cleaning/station_direct_match_mapping", unionByName=True)
# Check schema and partition columns
station_mapping_df.printSchema()
station_mapping_df.show(5)

root
 |-- station_key: string (nullable = true)
 |-- coord_key: string (nullable = true)
 |-- normalized_name: string (nullable = true)
 |-- canonical_station_id: string (nullable = true)
 |-- canonical_coord_key: string (nullable = true)
 |-- canonical_lat: double (nullable = true)
 |-- canonical_lon: double (nullable = true)
 |-- cluster_id: string (nullable = true)
 |-- cluster_size: long (nullable = true)
 |-- observed_trip_count: long (nullable = true)
 |-- first_year_seen: integer (nullable = true)
 |-- last_year_seen: integer (nullable = true)

+--------------------+-----------------+--------------------+--------------------+-------------------+-------------+-------------+--------------+------------+-------------------+---------------+--------------+
|         station_key|        coord_key|     normalized_name|canonical_station_id|canonical_coord_key|canonical_lat|canonical_lon|    cluster_id|cluster_size|observed_trip_count|first_year_seen|last_year_seen|
+--------------------+

In [29]:
station_mapping_df.count()

1901

In [4]:
rides_df.count()

27588048

In [ ]:
rides_df.createOrReplaceTempView("rides")
# print out 1 row example to see a sample of the data
spark.sql("SELECT * FROM rides LIMIT 1").show(truncate=False)

+------------------------------------------+-----------------------------------------------+--------------------------+----------------------+----------------------+-----------------------+----------------------+-------------------------------------+--------------------+---------------------+-------------------+-------------------+--------------------------+---------------------+--------------------+--------------------+--------------------------+------------------------+---------+
|end_station_key                           |start_station_key                              |start_station_name        |start_station_district|start_station_latitude|start_station_longitude|end_station_name      |end_station_district                 |end_station_latitude|end_station_longitude|start_time_ms      |end_time_ms        |start_station_name_norm   |end_station_name_norm|start_coord_key     |end_coord_key       |start_canonical_station_id|end_canonical_station_id|ride_year|
+-------------------------

In [32]:
station_mapping_df.columns

['station_key',
 'coord_key',
 'normalized_name',
 'canonical_station_id',
 'canonical_coord_key',
 'canonical_lat',
 'canonical_lon',
 'cluster_id',
 'cluster_size',
 'observed_trip_count',
 'first_year_seen',
 'last_year_seen']

In [6]:
from pyspark.sql import functions as F

# Read data
systeminfo_df = spark.read.option("header", "true").csv("./csv_exports/system_information.csv")
status_df = spark.read.option("header", "true").csv("./csv_exports/station_status.csv")
stationinfo_df = spark.read.option("header", "true").csv("./csv_exports/stations.csv")

print(systeminfo_df.columns)
print(status_df.columns)
print(stationinfo_df.columns)

['system_id', 'language', 'name', 'short_name', 'operator', 'url', 'purchase_url', 'start_date', 'phone_number', 'email', 'timezone', 'license_url', 'last_updated']
['station_id', 'num_bikes_available', 'num_ebikes_available', 'num_bikes_disabled', 'num_docks_available', 'num_docks_disabled', 'is_installed', 'is_renting', 'is_returning', 'last_reported', 'vehicle_types_available', 'vehicle_docks_available', 'feed_last_updated']
['station_id', 'external_id', 'name', 'short_name', 'lat', 'lon', 'rental_methods', 'capacity', 'electric_bike_surcharge_waiver', 'is_charging', 'eightd_has_key_dispenser', 'has_kiosk', 'last_updated']


In [7]:
stationinfo_df.createOrReplaceTempView("stations")
status_df.createOrReplaceTempView("status")
systeminfo_df.createOrReplaceTempView("systems")
# print out 1 row example to see a sample of the data
spark.sql("SELECT * FROM stations LIMIT 1").show(truncate=False)
spark.sql("SELECT * FROM status LIMIT 1").show(truncate=False)

+----------+------------------------------------+-------------------------+----------+----------------+----------------+----------------+----------+------------------------------+-----------+------------------------+---------+------------+
|station_id|external_id                         |name                     |short_name|lat             |lon             |rental_methods  |capacity  |electric_bike_surcharge_waiver|is_charging|eightd_has_key_dispenser|has_kiosk|last_updated|
+----------+------------------------------------+-------------------------+----------+----------------+----------------+----------------+----------+------------------------------+-----------+------------------------+---------+------------+
|1         |0b0fda98-08f3-11e7-a1cb-3863bb33a4e4|Drummond / de Maisonneuve|6001      |45.4996545106653|-73.576335310936|"[""CREDITCARD""| ""KEY""]"|27                            |0          |0                       |0        |1           |
+----------+----------------------------

In [35]:
stationinfo_df.explain(True)

== Parsed Logical Plan ==
UnresolvedDataSource format: csv, isStreaming: false, paths: 1 provided

== Analyzed Logical Plan ==
station_id: string, external_id: string, name: string, short_name: string, lat: string, lon: string, rental_methods: string, capacity: string, electric_bike_surcharge_waiver: string, is_charging: string, eightd_has_key_dispenser: string, has_kiosk: string, last_updated: string
Relation [station_id#1086,external_id#1087,name#1088,short_name#1089,lat#1090,lon#1091,rental_methods#1092,capacity#1093,electric_bike_surcharge_waiver#1094,is_charging#1095,eightd_has_key_dispenser#1096,has_kiosk#1097,last_updated#1098] csv

== Optimized Logical Plan ==
Relation [station_id#1086,external_id#1087,name#1088,short_name#1089,lat#1090,lon#1091,rental_methods#1092,capacity#1093,electric_bike_surcharge_waiver#1094,is_charging#1095,eightd_has_key_dispenser#1096,has_kiosk#1097,last_updated#1098] csv

== Physical Plan ==
FileScan csv [station_id#1086,external_id#1087,name#1088,sho

In [9]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType
import math

# ============================================================================
# 1. Load DataFrames (assuming they are already in Spark)
# ============================================================================
# station_mapping_df – from your silver layer
station_mapping_df = spark.read.parquet(
    "../../data/silver/station_cleaning/station_direct_match_mapping",
    unionByName=True
)

# stationinfo_df – from GBFS station_information (CSV or table)
stationinfo_df = spark.table("stations")  # adjust if needed

# status_df – from GBFS station_status (CSV or table)
status_df = spark.table("status")            # adjust if needed

# rides_df – from your rides table
rides_df = spark.table("rides")              # adjust if needed

# ============================================================================
# 2. Build mapping: canonical_station_id -> station_key (GBFS station_id)
# ============================================================================
mapping = station_mapping_df.select(
    F.col("canonical_station_id"),
    F.col("station_key")
).distinct()

# ============================================================================
# 3. Compute connectivity (total trips per station)
# ============================================================================
# Unify all trips (start and end) as station entries
starts = rides_df.select(F.col("start_canonical_station_id").alias("canonical_station_id"))
ends   = rides_df.select(F.col("end_canonical_station_id").alias("canonical_station_id"))
all_trips = starts.union(ends)

# Count trips per canonical_station_id
trip_counts = all_trips.groupBy("canonical_station_id") \
                       .count() \
                       .withColumnRenamed("count", "total_trips")

# Join with mapping to get station_key
connectivity = trip_counts.join(mapping, on="canonical_station_id", how="inner") \
                          .select("station_key", "total_trips")

# ============================================================================
# 4. Get live status (latest status per station, filter live)
# ============================================================================
# Cast necessary columns
# status_df = status_df.withColumn("is_installed", F.col("is_installed").cast("int")) \
#                      .withColumn("is_renting", F.col("is_renting").cast("int")) \
#                      .withColumn("feed_last_updated", F.col("feed_last_updated").cast("long"))

status_df = status_df.withColumn("is_installed", 
                                 F.regexp_extract(F.col("is_installed"), r"(\d+)", 1).cast("int")) \
                     .withColumn("is_renting", 
                                 F.regexp_extract(F.col("is_renting"), r"(\d+)", 1).cast("int")) \
                     .withColumn("feed_last_updated", 
                                 F.regexp_extract(F.col("feed_last_updated"), r"(\d+)", 1).cast("long"))
# Window to get the latest record per station
window_spec = Window.partitionBy("station_id").orderBy(F.col("feed_last_updated").desc())
latest_status = status_df.withColumn("rn", F.row_number().over(window_spec)) \
                         .filter("rn = 1") \
                         .drop("rn")

# Live stations: installed AND renting
live_stations = latest_status.filter(
    (F.col("is_installed") == 1) & (F.col("is_renting") == 1)
).select(F.col("station_id").alias("station_key")).distinct().withColumn("is_live", F.lit(1))
# ============================================================================
# 5. Combine all station data: coordinates, connectivity, live flag
# ============================================================================
# Use stationinfo_df for coordinates (lat, lon)
stations = stationinfo_df.select(
    F.col("station_id").alias("station_key"),
    F.col("lat").cast("double"),
    F.col("lon").cast("double")
)

# Join with connectivity (left join, fill 0 for stations with no trips)
stations = stations.join(connectivity, on="station_key", how="left") \
                   .fillna(0, subset=["total_trips"])

# Join with live flag (left join, fill 0)
stations = stations.join(live_stations, on="station_key", how="left") \
                   .fillna(0, subset=["is_live"])

# Optionally, filter out stations with missing coordinates (needed for proximity)
stations = stations.filter(F.col("lat").isNotNull() & F.col("lon").isNotNull())

# ============================================================================
# 6. Compute Proximity Score (average distance to all other stations)
# ============================================================================
# Haversine distance UDF
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # km
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = (math.sin(dlat/2)**2 +
         math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) *
         math.sin(dlon/2)**2)
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
    return R * c

haversine_udf = F.udf(haversine, DoubleType())

# Self‑join to get all pairs (distinct stations)
pairs = stations.alias("a").join(
    stations.alias("b"),
    F.col("a.station_key") != F.col("b.station_key")
)

pairs = pairs.withColumn(
    "distance_km",
    haversine_udf(F.col("a.lat"), F.col("a.lon"), F.col("b.lat"), F.col("b.lon"))
)

# Average distance per station
avg_dist = pairs.groupBy("a.station_key").agg(F.avg("distance_km").alias("avg_distance_km"))

# Proximity score: closer = higher score (0‑1)
proximity = avg_dist.withColumn(
    "proximity_score",
    1 / (1 + F.col("avg_distance_km"))
)

# ============================================================================
# 7. Normalize Connectivity Score (min‑max scaling)
# ============================================================================
# Note: we need to compute min/max from the stations DataFrame after joining with proximity
# But we'll compute separately and then join
conn_stats = stations.agg(F.min("total_trips").alias("min_trips"),
                          F.max("total_trips").alias("max_trips")).first()
min_trips, max_trips = conn_stats[0], conn_stats[1]

if max_trips == min_trips:
    connectivity_score = stations.withColumn("connectivity_score", F.lit(0.5))
else:
    connectivity_score = stations.withColumn(
        "connectivity_score",
        (F.col("total_trips") - min_trips) / (max_trips - min_trips)
    )

# ============================================================================
# 8. Combine all scores into a single DataFrame
# ============================================================================
combined = stations.select("station_key", "is_live") \
                   .join(connectivity_score.select("station_key", "connectivity_score"), on="station_key") \
                   .join(proximity.select("station_key", "proximity_score"), on="station_key")

# Define weights (adjust as needed)
w_prox = 0.4
w_conn = 0.4
w_avail = 0.2

combined = combined.withColumn(
    "total_score",
    w_prox * F.col("proximity_score") +
    w_conn * F.col("connectivity_score") +
    w_avail * F.col("is_live")
)

# ============================================================================
# 9. Select Top 10 Candidate Stations
# ============================================================================
top10 = combined.orderBy(F.col("total_score").desc()).limit(10)

top10.select(
    "station_key",
    "proximity_score",
    "connectivity_score",
    "is_live",
    "total_score"
).show(truncate=False)

+-----------+-------------------+------------------+-------+-------------------+
|station_key|proximity_score    |connectivity_score|is_live|total_score        |
+-----------+-------------------+------------------+-------+-------------------+
|621        |0.2375575268286857 |0.5               |1      |0.4950230107314743 |
|144        |0.2374798954002242 |0.5               |1      |0.49499195816008973|
|216        |0.23741110690311798|0.5               |1      |0.49496444276124724|
|233        |0.2373174454711988 |0.5               |1      |0.4949269781884795 |
|179        |0.23713712688704927|0.5               |1      |0.4948548507548197 |
|915        |0.23703400670895225|0.5               |1      |0.49481360268358093|
|619        |0.23687180630304452|0.5               |1      |0.4947487225212178 |
|43         |0.2368596434998169 |0.5               |1      |0.49474385739992677|
|209        |0.23679681387758034|0.5               |1      |0.4947187255510322 |
|174        |0.2367419325022

In [11]:
# ============================================================================
# 9. Select Top 10 Candidate Stations
# ============================================================================
top10 = combined.orderBy(F.col("total_score").desc()).limit(10)

# ---- Add reverse mapping ----
mapping_rev = mapping.select("station_key", "canonical_station_id").distinct()
top10_mapped = top10.join(mapping_rev, on="station_key", how="left")

# Show with canonical_station_id
top10_mapped.select(
    "station_key",
    "canonical_station_id",
    "proximity_score",
    "connectivity_score",
    "is_live",
    "total_score"
).show(truncate=False)

+-----------+--------------------+-------------------+------------------+-------+-------------------+
|station_key|canonical_station_id|proximity_score    |connectivity_score|is_live|total_score        |
+-----------+--------------------+-------------------+------------------+-------+-------------------+
|621        |NULL                |0.2375575268286857 |0.5               |1      |0.4950230107314743 |
|144        |NULL                |0.2374798954002242 |0.5               |1      |0.49499195816008973|
|216        |NULL                |0.23741110690311798|0.5               |1      |0.49496444276124724|
|233        |NULL                |0.2373174454711988 |0.5               |1      |0.4949269781884795 |
|179        |NULL                |0.23713712688704927|0.5               |1      |0.4948548507548197 |
|915        |NULL                |0.23703400670895225|0.5               |1      |0.49481360268358093|
|619        |NULL                |0.23687180630304452|0.5               |1      |0